In [ ]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget ${PREFIX}/01-agentic-rag/code/ingest.py
!wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

In [ ]:
# ${PREFIX} (Dollar Sign + Brackets): This tells the system to look for an Environment/Bash variable named PREFIX. 
# Because you defined PREFIX as a regular Python variable, the Bash environment does not know it exists. 
# It reads it as an empty value, causing the download to fail.
# Define it as an environment variable instead of a Python variable
%env PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

!wget ${PREFIX}/01-agentic-rag/code/ingest.py
!wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py


In [2]:
# {PREFIX} (Curly Brackets): This tells Google Colab to look for a Python variable named PREFIX. 
# Since you defined PREFIX = "..." as a Python variable on the first line, 
# Colab finds it and inserts the URL correctly
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/ingest.py
!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py


--2026-07-12 16:40:29--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-12 16:40:29 (35.9 MB/s) - ‘ingest.py’ saved [738/738]

--2026-07-12 16:40:29--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

In [4]:
%pip install minsearch

##### Loading the documents

In [5]:
from ingest import load_faq_data
documents = load_faq_data()
len(documents)

1375

In [6]:
# Generate questions only for the LLM Zoomcamp FAQ
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [7]:
documents = documents_llm
len(documents)

113

In [8]:
# Each document already has an id field:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


##### Generating questions with structured output

In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
import os

In [ ]:
from openai import OpenAI

# Resolve API key from environment if available (OPENAI_API_KEY preferred, fallback to GROQ_API_KEY)
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or GROQ_API_KEY environment variable before creating OpenAI client.")
openai_client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1") or os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
)

In [14]:
# Prepare the document as JSON:
import json
user_prompt = json.dumps(doc)

In [24]:
user_prompt

In [15]:
# Create the messages:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [16]:
response = openai_client.responses.parse(
    model="openai/gpt-oss-20b",
    input=messages,
    text_format=Questions
)

In [17]:
print(response)

ParsedResponse[TypeVar](id='resp_01kxbkcf2ef4kasn89mrgm2nx5', created_at=1783874600.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='openai/gpt-oss-20b', object='response', output=[ResponseReasoningItem(id='resp_01kxbkcf2ef4krttr9sw7ysbkg', summary=[], type='reasoning', content=[Content(text='We need to produce 5 questions that a student might ask based on the FAQ record. The record contains question: "I just discovered the course. Can I still join?" with answer: "Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions."\n\nWe need to formulate 5 questions that this student might ask, using as few words as possible from the record. They should be complete and not too short. We should not use too many words from the record. Use compact JSON format.\n\nWe must include all fields in the schema: "questions": array of string. That’s all. We need to output JSON with a single key "questions" and array

In [18]:
print(response.output_parsed)

questions=['Can I still join the course if I just found it?', 'Will I be able to get a certificate if I enroll now?', 'Do I need to submit a project to receive a certificate?', 'How long is the project submission window?', 'Is there a deadline for project submissions for late enrollees?']


In [19]:
# The parsed object is available in response.output_parsed
response.output_parsed.questions

  

['Can I still join the course if I just found it?',
 'Will I be able to get a certificate if I enroll now?',
 'Do I need to submit a project to receive a certificate?',
 'How long is the project submission window?',
 'Is there a deadline for project submissions for late enrollees?']

#### Reusable utilities

In [20]:
from evaluation_utils import llm_structured

In [21]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model="openai/gpt-oss-20b"
)

print(result.questions)

['Is it too late to enroll in the course now?', 'Do I still get a certificate if I join late?', 'When is the deadline to submit my project for a certificate?', 'Can I join the course after the project submission period has closed?', 'What steps do I need to take to receive a certificate after enrolling?']


In [22]:
usage.input_tokens, usage.output_tokens

(336, 343)

In [23]:
from evaluation_utils import calc_price
cost = calc_price(usage)
cost

{'input_cost': 0.000252, 'output_cost': 0.0015435, 'total_cost': 0.0017955}

In [24]:
# New pricing per 1 million tokens
input_price_per_million = 0.075
output_price_per_million = 0.30

input_cost = (usage.input_tokens / 1_000_000) * input_price_per_million
output_cost = (usage.output_tokens / 1_000_000) * output_price_per_million
total_cost = input_cost + output_cost

input_cost, output_cost, total_cost


(2.52e-05, 0.0001029, 0.0001281)

In [25]:
# Convert the questions into a list of records, each containing the question and the document id
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to enroll in the course now?',
  'document': '74eb249bbf'},
 {'question': 'Do I still get a certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'When is the deadline to submit my project for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I join the course after the project submission period has closed?',
  'document': '74eb249bbf'},
 {'question': 'What steps do I need to take to receive a certificate after enrolling?',
  'document': '74eb249bbf'}]

#### Generating Ground Truth for All Documents

In [26]:
from evaluation_utils import llm_structured_retry

In [27]:
# Generate questions into list of records that contain question and document id 
# for a single document and return the results along with usage information
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="openai/gpt-oss-20b"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [28]:
results = generate_ground_truth(doc)
results

([{'question': 'Is it still possible to enroll after the course launch?',
   'document': '74eb249bbf'},
  {'question': 'Can I join the class now and still earn a certificate?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to get certified if I sign up late?',
   'document': '74eb249bbf'},
  {'question': 'Will late registrations be accepted for the project deadline?',
   'document': '74eb249bbf'},
  {'question': 'How can I make sure my late entry counts for the course certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=336, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=497, output_tokens_details=OutputTokensDetails(reasoning_tokens=414), total_tokens=833))

In [29]:
len(documents)

113

In [30]:
from tqdm.auto import tqdm
# convert the question into a list of recores that contain question and document id 
# for all documents and return the results along with usage information
ground_truth = []
usages = []
# This works, but it runs one LLM call after another. Running it for all documents this way would take too long.
for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [31]:
len(ground_truth)

25

#### Parallel processing

In [32]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
# Use a thread pool to generate questions for multiple documents in parallel
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents[:15], generate_ground_truth)

  0%|          | 0/15 [00:00<?, ?it/s]

In [35]:
len(results)

15

In [40]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

75

In [ ]:
ground_truth

In [ ]:
usages

In [42]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.0384975

In [43]:
# Put the cost calculation function `calc_price(usage)` 
# into a function call `calc_total_price` in this file evaluation_utils.py and call it here
# to calculate the total cost for all usages
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.0384975

In [44]:
# To create a DataFrame from the ground truth records and save it to a CSV file
import pandas as pd 

In [46]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth

,question,document
0,Can I enroll in the course now that I've found...,74eb249bbf
1,What do I need to do to earn a certificate if ...,74eb249bbf
2,Is there still time to submit my project for a...,74eb249bbf
3,Do I have to finish the project to receive a c...,74eb249bbf
4,Will late applicants still have access to the ...,74eb249bbf
...,...,...
70,I noticed my FAQ dataset has fewer records tha...,e2d595f23c
71,The number of FAQ entries I pull now doesn’t m...,e2d595f23c
72,Does the FAQ data get updated or pruned after ...,e2d595f23c
73,Are there any guarantees about the stability o...,e2d595f23c


In [47]:
# Because the official Colab extension for VS Code is a remote kernel connection 
# that does not display Colab's internal cloud file tree in the IDE sidebar. 
df_ground_truth.to_csv("ground_truth-new.csv", index=False)

In [54]:
from google.colab import files
files.download('ground_truth-new.csv') 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Method 2: Convert the file to an HTML LinkYou can convert the CSV file 
# into an embedded, click-to-download hyperlink inside your notebook using pure Python.
import base64
from IPython.display import HTML

# Read the file data
with open('/content/ground_truth-new.csv', 'rb') as f:
    csv_bytes = f.read()

# Encode to a base64 string
b64 = base64.b64encode(csv_bytes).decode()

# Render a clickable HTML download link
html_link = f'<a href="data:file/csv;base64,{b64}" download="ground_truth-new.csv">👉 Click Here to Download Your CSV File 👈</a>'
HTML(html_link)


In [48]:
# Since your VS Code notebook is connected to the Colab runtime kernel, 
# you can use Python libraries to load and print the file data right inside your editor.
with open("ground_truth-new.csv", "r") as f:
    print("".join(f.readlines()))  # Displays the first 10 rows


question,document
Can I enroll in the course now that I've found it?,74eb249bbf
What do I need to do to earn a certificate if I join late?,74eb249bbf
Is there still time to submit my project for a certificate?,74eb249bbf
Do I have to finish the project to receive a certificate?,74eb249bbf
Will late applicants still have access to the course materials?,74eb249bbf
Do I need to wait for a confirmation email before I can start the LLM Zoomcamp?,977bf7786c
Can I begin the course and submit homework right after registering even if I haven't received any email?,977bf7786c
"Is registering for the LLM Zoomcamp mandatory, or can I join the classes without filling out the form?",977bf7786c
What does it mean if I register after the enrollment form has closed?,977bf7786c
How can I confirm that I'm accepted into the LLM Zoomcamp?,977bf7786c
Where can I find the live link for Office Hours or workshop sessions?,489dd1c9d9
"If I don’t have a Zoom link, how do I join the live session?",489dd1c9d9
How do

In [50]:
# To see it as a formatted table, use pandas
df = pd.read_csv("ground_truth-new.csv")
display(df)


,question,document
0,Can I enroll in the course now that I've found...,74eb249bbf
1,What do I need to do to earn a certificate if ...,74eb249bbf
2,Is there still time to submit my project for a...,74eb249bbf
3,Do I have to finish the project to receive a c...,74eb249bbf
4,Will late applicants still have access to the ...,74eb249bbf
...,...,...
70,I noticed my FAQ dataset has fewer records tha...,e2d595f23c
71,The number of FAQ entries I pull now doesn’t m...,e2d595f23c
72,Does the FAQ data get updated or pruned after ...,e2d595f23c
73,Are there any guarantees about the stability o...,e2d595f23c
